In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import xarray
import dask
import fsspec

import psutil
import time

In [2]:
nworkers = [8, 4, 2, 1]
nworkers = [4]
runs = 1
results = []

def measure(op, name, nworkers, run):
    start_net = psutil.net_io_counters()
    start_time = time.time()

    op.compute(num_workers=nworkers, scheduler="processes")

    end_time = time.time()
    end_net = psutil.net_io_counters()

    result = {
        "name": name,
        "run": run,
        "time": end_time-start_time,
        "bytes_recv": end_net.bytes_recv-start_net.bytes_recv,
        "bytes_sent": end_net.bytes_sent-start_net.bytes_sent,
        "packets_recv": end_net.packets_recv-start_net.packets_recv,
        "packets_sent": end_net.packets_sent-start_net.packets_sent,
        "errin": end_net.errin-start_net.errin,
        "errout": end_net.errout-start_net.errout,
        "dropin": end_net.dropin-start_net.dropin,
        "dropout": end_net.dropout-start_net.dropout,
        "workers": nworkers
    }

    return result

In [3]:
df = pd.read_csv("../../data_inventory.csv")
subset = df.query('type == "zarr" & variable == "t" & project == "CMIP6" & frequency == "mon" & experiment == "ssp585"')
location = subset["location"].iloc[0]
location

'https://api.cloud.ifca.es:8080/swift/v1/IPCC-IA-Monthly/zarr/CMIP6/ssp585/t_CMIP6_ssp585_mon_201501-210012'

In [ ]:
for w in nworkers:
    for r in range(runs):
        ds = xarray.open_zarr(location).chunk(member=-1, time=1)
        op = ds["t"].mean(["lat", "lon", "member"])
        results.append(measure(op, "zarr", w, r))

In [6]:
df = pd.read_csv("../../data_inventory.csv")
subset = df.query('type == "h5co" & variable == "t" & project == "CMIP6" & frequency == "mon" & experiment == "ssp585"')
location = subset["location"].iloc[0]
location

'https://api.cloud.ifca.es:8080/swift/v1/IPCC-IA-Monthly/HDF5CO/CMIP6/ssp585/t_CMIP6_ssp585_mon_201501-210012.nc'

In [ ]:
for w in nworkers:
    for r in range(runs):
        fs = fsspec.open(location, cache_type="first", block_size=4*2**20)
        ds = xarray.open_dataset(
            fs.open(),
            engine="h5netcdf",
            driver_kwds={
                "fs_page_size": 4*2**20,
                "page_buf_size": 100*2**20})
        
        ds = xarray.open_dataset(location, engine="h5netcdf").chunk(member=-1, time=1)
        op = ds["t"].mean(["lat", "lon", "member"])
        results.append(measure(op, "h5co", w, r))

In [7]:
df = pd.read_csv("../../data_inventory.csv")
subset = df.query('type == "opendap" & variable == "t" & project == "CMIP6" & frequency == "mon" & experiment == "ssp585"')
location = subset["location"].iloc[0]
location

'https://hub.climate4r.ifca.es/thredds/dodsC/ipcc/ar6/atlas/ia-monthly/CMIP6/ssp585/t_CMIP6_ssp585_mon_201501-210012.nc'

In [8]:
!sed -i 's|DEFLATE=0|DEFLATE=1|' ~/.dodsrc && cat ~/.dodsrc | grep DEFLATE

HTTP.DEFLATE=1


In [3]:
!sed -i 's|DEFLATE=1|DEFLATE=0|' ~/.dodsrc && cat ~/.dodsrc | grep DEFLATE

HTTP.DEFLATE=0


In [18]:
for w in nworkers:
    for r in range(runs):
        ds = xarray.open_dataset(location).chunk({"member":35, "time": 10, "lat": 90, "lon": 180})
        op = ds["t"].mean(["lat", "lon", "member"])
        results.append(measure(op, "opendap-deflate2", w, r))

In [4]:
ds = xarray.open_zarr("https://api.cloud.ifca.es:8080/swift/v1/IPCC-IA-Monthly/zarr/CMIP6/ssp585/t_CMIP6_ssp585_mon_201501-210012").chunk(member=-1, time=1)
ds["t"]

<xarray.DataArray 't' (member: 34, time: 1032, lat: 180, lon: 360)> Size: 9GB
dask.array<rechunk-merge, shape=(34, 1032, 180, 360), dtype=float32, chunksize=(34, 1, 180, 360), chunktype=numpy.ndarray>
Coordinates:
    height2m  float64 8B ...
  * lat       (lat) float64 1kB -89.5 -88.5 -87.5 -86.5 ... 86.5 87.5 88.5 89.5
  * lon       (lon) float64 3kB -179.5 -178.5 -177.5 ... 177.5 178.5 179.5
  * member    (member) <U42 6kB 'CSIRO-ARCCSS_ACCESS-CM2_r1i1p1f1' ... 'MOHC_...
  * time      (time) datetime64[ns] 8kB 2015-01-01 2015-02-01 ... 2100-12-01
Attributes:
    cell_methods:   time: mean within days time: mean over days area: mean
    comment:        Monthly mean of daily mean near-surface (usually, 2 meter...
    grid_mapping:   crs
    long_name:      Monthly mean of daily mean temperature
    standard_name:  air_temperature
    units:          degC

## 8Mib chunks

In [4]:
ds = xarray.open_zarr("https://api.cloud.ifca.es:8080/swift/v1/tests/gpfs/ces/share-7c11c2a4-9d9f-40f5-b95e-396bcbf3f608/IPCC-HUB/data/t_CMIP6_historical_mon_185001-201412")
ds["t"]

<xarray.DataArray 't' (member: 35, time: 1980, lat: 180, lon: 360)> Size: 18GB
dask.array<open_dataset-t, shape=(35, 1980, 180, 360), dtype=float32, chunksize=(35, 1, 180, 360), chunktype=numpy.ndarray>
Coordinates:
    height2m  float64 8B ...
  * lat       (lat) float64 1kB -89.5 -88.5 -87.5 -86.5 ... 86.5 87.5 88.5 89.5
  * lon       (lon) float64 3kB -179.5 -178.5 -177.5 ... 177.5 178.5 179.5
  * member    (member) <U45 6kB 'CSIRO-ARCCSS_ACCESS-CM2_r1i1p1f1' ... 'MOHC_...
  * time      (time) datetime64[ns] 16kB 1850-01-01 1850-02-01 ... 2014-12-01
Attributes:
    cell_methods:   time: mean within days time: mean over days area: mean
    comment:        Monthly mean of daily mean near-surface (usually, 2 meter...
    grid_mapping:   crs
    long_name:      Monthly mean of daily mean temperature
    standard_name:  air_temperature
    units:          degC

In [5]:
for w in nworkers:
    for r in range(runs):
        op = ds["t"].mean(["lat", "lon", "member"])
        results.append(measure(op, "zarr-spatial", w, r))

In [6]:
results

[{'name': 'zarr-spatial',
  'run': 0,
  'time': 149.38797044754028,
  'bytes_recv': 5170976512,
  'bytes_sent': 22336612,
  'packets_recv': 3903038,
  'packets_sent': 323719,
  'errin': 0,
  'errout': 0,
  'dropin': 0,
  'dropout': 0,
  'workers': 4}]

In [7]:
5170976512 / 2**20

4931.427490234375

# Split spatial chunks

In [8]:
ds = xarray.open_zarr("https://api.cloud.ifca.es:8080/swift/v1/tests/test")
ds["t"]

<xarray.DataArray 't' (member: 35, time: 1980, lat: 180, lon: 360)> Size: 18GB
dask.array<open_dataset-t, shape=(35, 1980, 180, 360), dtype=float32, chunksize=(35, 1, 90, 180), chunktype=numpy.ndarray>
Coordinates:
    height2m  float64 8B ...
  * lat       (lat) float64 1kB -89.5 -88.5 -87.5 -86.5 ... 86.5 87.5 88.5 89.5
  * lon       (lon) float64 3kB -179.5 -178.5 -177.5 ... 177.5 178.5 179.5
  * member    (member) <U45 6kB 'CSIRO-ARCCSS_ACCESS-CM2_r1i1p1f1' ... 'MOHC_...
  * time      (time) datetime64[ns] 16kB 1850-01-01 1850-02-01 ... 2014-12-01
Attributes:
    cell_methods:   time: mean within days time: mean over days area: mean
    comment:        Monthly mean of daily mean near-surface (usually, 2 meter...
    grid_mapping:   crs
    long_name:      Monthly mean of daily mean temperature
    standard_name:  air_temperature
    units:          degC

In [9]:
for w in nworkers:
    for r in range(runs):
        op = ds["t"].mean(["lat", "lon", "member"])
        results.append(measure(op, "zarr-subspatial", w, r))

In [10]:
results

[{'name': 'zarr-spatial',
  'run': 0,
  'time': 149.38797044754028,
  'bytes_recv': 5170976512,
  'bytes_sent': 22336612,
  'packets_recv': 3903038,
  'packets_sent': 323719,
  'errin': 0,
  'errout': 0,
  'dropin': 0,
  'dropout': 0,
  'workers': 4},
 {'name': 'zarr-subspatial',
  'run': 0,
  'time': 219.5062234401703,
  'bytes_recv': 5274268902,
  'bytes_sent': 23830043,
  'packets_recv': 3993663,
  'packets_sent': 330737,
  'errin': 0,
  'errout': 0,
  'dropin': 0,
  'dropout': 0,
  'workers': 4}]

In [13]:
(5274268902 - 5170976512) / 2**20

98.50729942321777

# Group time values

In [4]:
ds = xarray.open_zarr("https://api.cloud.ifca.es:8080/swift/v1/IPCC-IA-Monthly/test2")
ds["t"]

<xarray.DataArray 't' (member: 35, time: 1980, lat: 180, lon: 360)> Size: 18GB
dask.array<open_dataset-t, shape=(35, 1980, 180, 360), dtype=float32, chunksize=(35, 10, 90, 180), chunktype=numpy.ndarray>
Coordinates:
    height2m  float64 8B ...
  * lat       (lat) float64 1kB -89.5 -88.5 -87.5 -86.5 ... 86.5 87.5 88.5 89.5
  * lon       (lon) float64 3kB -179.5 -178.5 -177.5 ... 177.5 178.5 179.5
  * member    (member) <U45 6kB 'CSIRO-ARCCSS_ACCESS-CM2_r1i1p1f1' ... 'MOHC_...
  * time      (time) datetime64[ns] 16kB 1850-01-01 1850-02-01 ... 2014-12-01
Attributes:
    cell_methods:   time: mean within days time: mean over days area: mean
    comment:        Monthly mean of daily mean near-surface (usually, 2 meter...
    grid_mapping:   crs
    long_name:      Monthly mean of daily mean temperature
    standard_name:  air_temperature
    units:          degC

In [5]:
for w in nworkers:
    for r in range(runs):
        op = ds["t"].mean(["lat", "lon", "member"])
        results.append(measure(op, "zarr-subspatial", w, r))

In [19]:
results

[{'name': 'zarr-subspatial',
  'run': 0,
  'time': 135.67295956611633,
  'bytes_recv': 5204675537,
  'bytes_sent': 22432252,
  'packets_recv': 3937505,
  'packets_sent': 324423,
  'errin': 0,
  'errout': 0,
  'dropin': 0,
  'dropout': 0,
  'workers': 4},
 {'name': 'opendap-deflate',
  'run': 0,
  'time': 174.64632630348206,
  'bytes_recv': 5371459126,
  'bytes_sent': 39177964,
  'packets_recv': 4045389,
  'packets_sent': 551758,
  'errin': 0,
  'errout': 0,
  'dropin': 0,
  'dropout': 0,
  'workers': 4},
 {'name': 'opendap-deflate2',
  'run': 0,
  'time': 98.56832456588745,
  'bytes_recv': 5354708361,
  'bytes_sent': 35581734,
  'packets_recv': 4134613,
  'packets_sent': 524794,
  'errin': 0,
  'errout': 0,
  'dropin': 0,
  'dropout': 0,
  'workers': 4},
 {'name': 'opendap-deflate2',
  'run': 0,
  'time': 149.16110682487488,
  'bytes_recv': 5391683297,
  'bytes_sent': 36006680,
  'packets_recv': 4142746,
  'packets_sent': 523763,
  'errin': 0,
  'errout': 0,
  'dropin': 0,
  'dropout':

In [17]:
(5354708361 - 5204675537) / 2**20

143.08245086669922

# Big chunk (meses mas tarde de lo anterior)

In [14]:
ds = xarray.open_zarr("https://api.cloud.ifca.es:8080/swift/v1/IPCC-IA-Monthly/test-big-chunk")
ds["t"]

<xarray.DataArray 't' (member: 35, time: 1980, lat: 180, lon: 360)> Size: 18GB
dask.array<open_dataset-t, shape=(35, 1980, 180, 360), dtype=float32, chunksize=(35, 120, 45, 90), chunktype=numpy.ndarray>
Coordinates:
    height2m  float64 8B ...
  * lat       (lat) float64 1kB -89.5 -88.5 -87.5 -86.5 ... 86.5 87.5 88.5 89.5
  * lon       (lon) float64 3kB -179.5 -178.5 -177.5 ... 177.5 178.5 179.5
  * member    (member) <U45 6kB 'CSIRO-ARCCSS_ACCESS-CM2_r1i1p1f1' ... 'MOHC_...
  * time      (time) datetime64[ns] 16kB 1850-01-01 1850-02-01 ... 2014-12-01
Attributes:
    cell_methods:   time: mean within days time: mean over days area: mean
    comment:        Monthly mean of daily mean near-surface (usually, 2 meter...
    grid_mapping:   crs
    long_name:      Monthly mean of daily mean temperature
    standard_name:  air_temperature
    units:          degC

In [15]:
for w in nworkers:
    for r in range(runs):
        op = ds["t"].mean(["lat", "lon", "member"])
        results.append(measure(op, "zarr-64m-chunk", w, r))

In [16]:
results

[{'name': 'zarr-spatial',
  'run': 0,
  'time': 149.38797044754028,
  'bytes_recv': 5170976512,
  'bytes_sent': 22336612,
  'packets_recv': 3903038,
  'packets_sent': 323719,
  'errin': 0,
  'errout': 0,
  'dropin': 0,
  'dropout': 0,
  'workers': 4},
 {'name': 'zarr-subspatial',
  'run': 0,
  'time': 219.5062234401703,
  'bytes_recv': 5274268902,
  'bytes_sent': 23830043,
  'packets_recv': 3993663,
  'packets_sent': 330737,
  'errin': 0,
  'errout': 0,
  'dropin': 0,
  'dropout': 0,
  'workers': 4},
 {'name': 'zarr-64m-chunk',
  'run': 0,
  'time': 124.90156602859497,
  'bytes_recv': 5143971744,
  'bytes_sent': 21874253,
  'packets_recv': 3885864,
  'packets_sent': 324277,
  'errin': 0,
  'errout': 0,
  'dropin': 0,
  'dropout': 0,
  'workers': 4}]